# Comparison of Embedding Models for Semantic Similarity

**Objective:** This notebook evaluates and compares the performance of different models for generating sentence embeddings, focusing on their ability to capture semantic similarity. A paraphrase dataset is used, where a good embedding model should assign high similarity scores between original sentences and their paraphrases.

**Dataset:** The [Humarin ChatGPT Paraphrases](https://huggingface.co/datasets/humarin/chatgpt-paraphrases) dataset is employed. It consists of pairs of sentences and their paraphrased versions, allowing quantitative assessment of how well embeddings preserve semantic meaning.

**Embedding Models:** 
- **`Qwen-3-0.6B`** – specialized embedding model from the Qwen series for semantic tasks.
- **`thenlper/gte-large`** – a large embedding model trained for general-purpose semantic similarity. 


**General Language Models for Embeddings:**  
- **`Qwen-3-0.6B`** 
 

**Evaluation Approach:**  
- **Cosine Similarity:** Cosine similarity is applied to compare embeddings. Higher cosine similarity (max 1) indicates that two sentences are semantically closer.  

**Goal:** The notebook aims to compare:  
1. Different models of varying sizes on semantic similarity tasks.  
2. Specialized embedding models versus general-purpose LLMs used to compute embeddings.  





# Requirements
- pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu129
- pip install pandas datasets sentence-transformers transformers numpy seaborn matplotlib scikit-learn
- pip install accelerate
- pip install faiss-cpu

### Setup and Import of Libraries


In [1]:
# Standard Python Libraryes 
import ast
import gc
import os
from functools import partial
from pathlib import Path
from typing import Optional, Tuple

# Numerical Computation and Data Management
import faiss
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D

# Machine Learning e NLP
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util
from sklearn.manifold import TSNE
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset
from sklearn.metrics.pairwise import cosine_similarity
from scipy import stats

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")





c:\Users\nlp-user\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


### Loading and preparation of the dataset


**Original dataset:** each row contains an `original sentence` and a list of its `paraphrases` stored as a string.  
**Transformation:** each paraphrase is paired with its original sentence, creating one row per (original, paraphrase) pair.  
**Final dataset:** columns are: `text` → original sentence ; `paraphrase` → a single paraphrase  ; `group_id` → identifier linking paraphrases from the same original sentence.



In [2]:
ds = load_dataset("humarin/chatgpt-paraphrases", split="train[:10000]")

def expand_fn(batch, indices):
    out = {"text": [], "paraphrase": [], "group_id": []}
    for idx, (t, ps_str) in zip(indices, zip(batch["text"], batch["paraphrases"])):
        try:
            ps = ast.literal_eval(ps_str)
        except Exception as e:
            print(f"Error evaluating row {idx}: {e}")
            ps = []
        for p in ps:
            out["text"].append(t)
            out["paraphrase"].append(p)
            out["group_id"].append(idx)  
    return out

exploded_ds = ds.map(
    expand_fn,
    with_indices=True,       
    batched=True,
    remove_columns=ds.column_names
)

print(f"Original examples: {len(ds)}, generated pairs: {len(exploded_ds)}")



Original examples: 10000, generated pairs: 50000


In [3]:
for i in range(15):
    print(exploded_ds[i])

{'text': 'What is the step by step guide to invest in share market in india?', 'paraphrase': 'Can you provide a detailed procedure for investing in the Indian stock market?', 'group_id': 0}
{'text': 'What is the step by step guide to invest in share market in india?', 'paraphrase': 'What are the sequential instructions for investing in shares in India?', 'group_id': 0}
{'text': 'What is the step by step guide to invest in share market in india?', 'paraphrase': 'Could you outline the step-by-step process for investing in the Indian share market?', 'group_id': 0}
{'text': 'What is the step by step guide to invest in share market in india?', 'paraphrase': 'What is the systematic guide to investing in the Indian stock exchange?', 'group_id': 0}
{'text': 'What is the step by step guide to invest in share market in india?', 'paraphrase': 'Can you provide a comprehensive guide on how to invest in the Indian share market?', 'group_id': 0}
{'text': 'What is the story of Kohinoor (Koh-i-Noor) Di

# Extra: Code for question retrieval using similarity with embeddings stored in FAISS.

In [4]:
def load_or_download_model(
    model_id: str,
    models_dir: Optional[Path] = None
) -> Optional[SentenceTransformer]:
    """
    Fetches a SentenceTransformer model, either from a local cache or by
    downloading from the Hugging Face Hub.

    If the model is not found locally, it is downloaded and saved to a structured
    directory. The model object is then returned, ready for encoding.

    Args:
        model_id (str): The model identifier from the Hugging Face Hub
                        (e.g., "all-MiniLM-L6-v2").
        models_dir (Optional[Path]): The root directory for storing models.
                                     If None, defaults to "models" in the CWD.

    Returns:
        Optional[SentenceTransformer]: The loaded SentenceTransformer model,
        or None if an error occurs.
    """
    if models_dir is None:
        models_dir = Path.cwd() / "models"

    sanitized_model_id = model_id.replace("/", "__")
    target_dir = models_dir / "sentence_transformer" / sanitized_model_id

    # If the model is cached locally, load it from the specified path
    if target_dir.exists() and (target_dir / "config.json").is_file():
        print(f"Loading SentenceTransformer model from local cache: {target_dir}")
        try:
            model = SentenceTransformer(str(target_dir))
            return model
        except Exception as e:
            print(f"Error loading model from cache: {e}")
            return None

    # If not cached, download from the Hub and then save it
    print(f"Downloading SentenceTransformer model from Hub: {model_id}")
    try:
        model = SentenceTransformer(model_id)
        print(f"Saving model to cache: {target_dir}")
        target_dir.mkdir(parents=True, exist_ok=True)
        model.save(str(target_dir))
        return model
    except Exception as e:
        print(f"Error downloading or saving model: {e}")
        return None

In [5]:
def load_or_download_hf_model(
    model_name: str,
    models_dir: str = "models",
    device: str = "cuda"
) -> tuple:
    """
    Load a HuggingFace model and tokenizer, caching locally.

    Args:
        model_name (str): HuggingFace model ID.
        models_dir (str): Directory to cache the model.
        device (str): "cpu" or "cuda".

    Returns:
        tuple: (model, tokenizer) loaded on the specified device
    """
    models_path = Path(models_dir) / model_name.replace("/", "__")
    models_path.mkdir(parents=True, exist_ok=True)

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=str(models_path))

    # Load model
    model = AutoModel.from_pretrained(model_name, cache_dir=str(models_path))
    model.to(device)

    return model, tokenizer

from transformers import AutoModel, AutoTokenizer


In [6]:
def compute_recall_multi(df_orig, df_para, top_k=5):
    dim = len(df_para.iloc[0]["emb_para"])
    xb = np.vstack(df_para["emb_para"].to_numpy()).astype("float32")
    faiss.normalize_L2(xb)
    index = faiss.IndexFlatIP(dim)
    index.add(xb)

    recalls = []
    for _, row in df_orig.iterrows():
        q = np.array(row["emb_orig"], dtype="float32").reshape(1, -1)
        faiss.normalize_L2(q)
        D, I = index.search(q, top_k)
        
        positives = set(df_para[df_para["group_id"] == row["group_id"]].index.tolist())
        retrieved = set(I[0].tolist())
        recalls.append(len(positives.intersection(retrieved)) / len(positives) if positives else 0.0)

    return {"Recall@{}".format(top_k): np.mean(recalls)}


def compute_metrics(df_para, df_orig, top_k_list=[1,3,5]):
    dim = len(df_orig.iloc[0]["emb_orig"])
    xb = np.vstack(df_orig["emb_orig"].to_numpy()).astype("float32")
    faiss.normalize_L2(xb)
    index = faiss.IndexFlatIP(dim)
    index.add(xb)

    rr_at_k = {k: [] for k in top_k_list}
    for _, row in df_para.iterrows():
        q = np.array(row["emb_para"], dtype="float32").reshape(1, -1)
        faiss.normalize_L2(q)
        D, I = index.search(q, max(top_k_list))
        retrieved_ids = df_orig.iloc[I[0]]["group_id"].tolist()  
        for k in top_k_list:
            top_k_ids = retrieved_ids[:k]
            if row["group_id"] in top_k_ids:  
                rank = top_k_ids.index(row["group_id"]) + 1
                rr_at_k[k].append(1.0 / rank)
            else:
                rr_at_k[k].append(0.0)

    return {f"MRR@{k}": np.mean(rr_at_k[k]) for k in top_k_list}



In [7]:
def get_mean_pooled_embedding(texts, model, tokenizer):
    model.eval()
    with torch.no_grad():
        toks = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(model.device)
        outputs = model(**toks)
        last_hidden = outputs.last_hidden_state
        mask = toks["attention_mask"].unsqueeze(-1).expand(last_hidden.size()).float()
        summed = torch.sum(last_hidden * mask, dim=1)
        count = torch.clamp(mask.sum(dim=1), min=1e-9)
        mean_pooled = summed / count        
        mean_pooled = F.normalize(mean_pooled, p=2, dim=1)
        
        return [e.cpu().numpy().astype("float32") for e in mean_pooled]


In [8]:
def compute_embeddings(df, text_col, model, tokenizer=None, batch_size=256, hf_model=True, emb_col=None):
    
    ds = Dataset.from_pandas(df)

    if hf_model:
        def batch_embed(batch):
            return {"emb": get_mean_pooled_embedding(batch[text_col], model, tokenizer)}
    else:
        def batch_embed(batch):
            
            embs = model.encode(batch[text_col], convert_to_tensor=False)
            
            embs = [np.array(e, dtype="float32") for e in embs]
            return {"emb": embs}

    
    ds_emb = ds.map(batch_embed, batched=True, batch_size=batch_size)

    
    df_emb = ds_emb.to_pandas()

    
    if emb_col is None:
        emb_col = f"emb_{text_col}"
    df_emb = df_emb.rename(columns={"emb": emb_col})

    return df_emb


In [9]:
# 1. Converti in dataframe
df = exploded_ds.to_pandas()

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

df_orig = df[["group_id", "text"]].drop_duplicates().reset_index(drop=True)

df_para = df[["group_id", "paraphrase"]].reset_index(drop=True)

print("Originals:")
print(df_orig.head())

print("\nParaphrases:")
print(df_para.head())



Originals:
   group_id  \
0         0   
1         1   
2         2   
3         3   
4         4   

                                                                           text  
0            What is the step by step guide to invest in share market in india?  
1                           What is the story of Kohinoor (Koh-i-Noor) Diamond?  
2     How can I increase the speed of my internet connection while using a VPN?  
3                            Why am I mentally very lonely? How can I solve it?  
4  Which one dissolve in water quikly sugar, salt, methane and carbon di oxide?  

Paraphrases:
   group_id  \
0         0   
1         0   
2         0   
3         0   
4         0   

                                                                             paraphrase  
0        Can you provide a detailed procedure for investing in the Indian stock market?  
1                What are the sequential instructions for investing in shares in India?  
2  Could you outline the step-b

In [10]:
results_df = pd.DataFrame(columns=[
    "model",
    "dim",
    "MRR@1_para_to_orig",
    "MRR@3_para_to_orig",
    "MRR@5_para_to_orig",
    "Recall@5_orig_to_para",
    "Mean_Cosine",              
    "Median_Cosine",            
    "Std_Dev_Cosine",           
    "Min_Cosine"                
])



In [11]:
embedding_complete_space = pd.DataFrame(columns=[
    "Model_Name",
    "emb_Mean_Cosine",      
    "emb_Median_Cosine",    
    "emb_Std_Dev_Cosine",   
    "emb_Min_Cosine",
    "emb_Max_Cosine",
    "emb_Q1_Cosine",
    "emb_Q3_Cosine"
])

In [12]:
# --- 1. Load specialized model ---
model_used = "thenlper/gte-large"
model_minilm = load_or_download_model(model_used)

df_orig_emb = compute_embeddings(df_orig, text_col="text", model=model_minilm, batch_size=256, hf_model=False, emb_col="emb_orig")
df_para_emb = compute_embeddings(df_para, text_col="paraphrase", model=model_minilm, batch_size=256, hf_model=False, emb_col="emb_para")


cos_sims = []
for _, para_row in df_para_emb.iterrows():
    orig_row = df_orig_emb[df_orig_emb["group_id"] == para_row["group_id"]].iloc[0]
    cos_sim = cosine_similarity(
        para_row["emb_para"].reshape(1, -1),
        orig_row["emb_orig"].reshape(1, -1)
    )[0,0]
    cos_sims.append(cos_sim)

mean_cosine_similarity = np.mean(cos_sims)
median_cosine_similarity = np.median(cos_sims)
std_dev_cosine_similarity = np.std(cos_sims)
min_cosine_similarity = np.min(cos_sims)
max_cosine_similarity = np.max(cos_sims)

metrics_mrr = compute_metrics(df_para_emb, df_orig_emb, top_k_list=[1,3,5])
metrics_recall = compute_recall_multi(df_orig_emb, df_para_emb, top_k=5)

results_df.loc[len(results_df)] = [
    model_used,
    len(df_orig_emb.iloc[0]["emb_orig"]),
    metrics_mrr["MRR@1"],
    metrics_mrr["MRR@3"],
    metrics_mrr["MRR@5"],
    metrics_recall["Recall@5"],
    mean_cosine_similarity,      
    median_cosine_similarity,    
    std_dev_cosine_similarity,   
    min_cosine_similarity        
]

all_orig_embs = np.vstack(df_orig_emb["emb_orig"].to_numpy())
all_para_embs = np.vstack(df_para_emb["emb_para"].to_numpy())

cosine_matrix = cosine_similarity(all_para_embs, all_orig_embs)
all_sim_scores = cosine_matrix.flatten()

embedding_complete_space.loc[len(embedding_complete_space)] = [
    model_used,
    np.mean(all_sim_scores),
    np.median(all_sim_scores),
    np.std(all_sim_scores),
    np.min(all_sim_scores),
    np.max(all_sim_scores),
    np.quantile(all_sim_scores, 0.25),
    np.quantile(all_sim_scores, 0.75)
]


Loading SentenceTransformer model from local cache: c:\Users\nlp-user\Desktop\completed_project\models\sentence_transformer\thenlper__gte-large


Map: 100%|██████████| 50000/50000 [03:21<00:00, 247.81 examples/s]


In [13]:
del model_minilm
gc.collect()
torch.cuda.empty_cache()

In [14]:
# --- 1. Load specialized model ---
model_used = "Qwen/Qwen3-Embedding-0.6B"

model_minilm = load_or_download_model(model_used)

df_orig_emb = compute_embeddings(df_orig, text_col="text", model=model_minilm, batch_size=256, hf_model=False, emb_col="emb_orig")
df_para_emb = compute_embeddings(df_para, text_col="paraphrase", model=model_minilm, batch_size=256, hf_model=False, emb_col="emb_para")


cos_sims = []
for _, para_row in df_para_emb.iterrows():
    orig_row = df_orig_emb[df_orig_emb["group_id"] == para_row["group_id"]].iloc[0]
    cos_sim = cosine_similarity(
        para_row["emb_para"].reshape(1, -1),
        orig_row["emb_orig"].reshape(1, -1)
    )[0,0]
    cos_sims.append(cos_sim)

mean_cosine_similarity = np.mean(cos_sims)
median_cosine_similarity = np.median(cos_sims)
std_dev_cosine_similarity = np.std(cos_sims)
min_cosine_similarity = np.min(cos_sims)
max_cosine_similarity = np.max(cos_sims)

metrics_mrr = compute_metrics(df_para_emb, df_orig_emb, top_k_list=[1,3,5])
metrics_recall = compute_recall_multi(df_orig_emb, df_para_emb, top_k=5)

results_df.loc[len(results_df)] = [
    model_used,
    len(df_orig_emb.iloc[0]["emb_orig"]),
    metrics_mrr["MRR@1"],
    metrics_mrr["MRR@3"],
    metrics_mrr["MRR@5"],
    metrics_recall["Recall@5"],
    mean_cosine_similarity,      
    median_cosine_similarity,    
    std_dev_cosine_similarity,   
    min_cosine_similarity        
]

all_orig_embs = np.vstack(df_orig_emb["emb_orig"].to_numpy())
all_para_embs = np.vstack(df_para_emb["emb_para"].to_numpy())

cosine_matrix = cosine_similarity(all_para_embs, all_orig_embs)
all_sim_scores = cosine_matrix.flatten()

embedding_complete_space.loc[len(embedding_complete_space)] = [
    model_used,
    np.mean(all_sim_scores),
    np.median(all_sim_scores),
    np.std(all_sim_scores),
    np.min(all_sim_scores),
    np.max(all_sim_scores),
    np.quantile(all_sim_scores, 0.25),
    np.quantile(all_sim_scores, 0.75)
]


Loading SentenceTransformer model from local cache: c:\Users\nlp-user\Desktop\completed_project\models\sentence_transformer\Qwen__Qwen3-Embedding-0.6B


Map: 100%|██████████| 50000/50000 [05:28<00:00, 152.17 examples/s]


In [15]:
del model_minilm
gc.collect()
torch.cuda.empty_cache()

In [16]:
model_used = "Qwen/Qwen3-0.6B"
model, tokenizer = load_or_download_hf_model(
    model_name=model_used,
    models_dir="models",
    device=device
)
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

df_orig_emb = compute_embeddings(df_orig, text_col="text", model=model, tokenizer=tokenizer, batch_size=256, hf_model=True, emb_col="emb_orig")
df_para_emb = compute_embeddings(df_para, text_col="paraphrase", model=model, tokenizer=tokenizer, batch_size=256, hf_model=True, emb_col="emb_para")

cos_sims = []
for _, para_row in df_para_emb.iterrows():
    orig_row = df_orig_emb[df_orig_emb["group_id"] == para_row["group_id"]].iloc[0]
    cos_sim = cosine_similarity(
        para_row["emb_para"].reshape(1, -1),
        orig_row["emb_orig"].reshape(1, -1)
    )[0,0]
    cos_sims.append(cos_sim)

mean_cosine_similarity = np.mean(cos_sims)
median_cosine_similarity = np.median(cos_sims)
std_dev_cosine_similarity = np.std(cos_sims)
min_cosine_similarity = np.min(cos_sims)
max_cosine_similarity = np.max(cos_sims)

metrics_mrr = compute_metrics(df_para_emb, df_orig_emb, top_k_list=[1,3,5])
metrics_recall = compute_recall_multi(df_orig_emb, df_para_emb, top_k=5)

results_df.loc[len(results_df)] = [
    model_used,
    len(df_orig_emb.iloc[0]["emb_orig"]),
    metrics_mrr["MRR@1"],
    metrics_mrr["MRR@3"],
    metrics_mrr["MRR@5"],
    metrics_recall["Recall@5"],
    mean_cosine_similarity,      
    median_cosine_similarity,    
    std_dev_cosine_similarity,   
    min_cosine_similarity        
]


all_orig_embs = np.vstack(df_orig_emb["emb_orig"].to_numpy())
all_para_embs = np.vstack(df_para_emb["emb_para"].to_numpy())

cosine_matrix = cosine_similarity(all_para_embs, all_orig_embs)
all_sim_scores = cosine_matrix.flatten()

embedding_complete_space.loc[len(embedding_complete_space)] = [
    model_used,
    np.mean(all_sim_scores),
    np.median(all_sim_scores),
    np.std(all_sim_scores),
    np.min(all_sim_scores),
    np.max(all_sim_scores),
    np.quantile(all_sim_scores, 0.25),
    np.quantile(all_sim_scores, 0.75)
]


Map: 100%|██████████| 50000/50000 [08:48<00:00, 94.69 examples/s] 


In [17]:
del model,tokenizer
gc.collect()
torch.cuda.empty_cache()

In [18]:
print(results_df)

                       model   dim  MRR@1_para_to_orig  MRR@3_para_to_orig  \
0         thenlper/gte-large  1024             0.95576            0.970813   
1  Qwen/Qwen3-Embedding-0.6B  1024             0.95602            0.970983   
2            Qwen/Qwen3-0.6B  1024             0.65816            0.698337   

   MRR@5_para_to_orig  Recall@5_orig_to_para  Mean_Cosine  Median_Cosine  \
0            0.972193                0.93804     0.958973       0.965250   
1            0.972294                0.93806     0.903549       0.915874   
2            0.705468                0.66728     0.952301       0.958187   

   Std_Dev_Cosine  Min_Cosine  
0        0.026975    0.754064  
1        0.059133    0.460187  
2        0.031589    0.300971  


### Evaluation of Embedding Models for Paraphrase Recognition

The table below reports the performance of three embedding models on a paraphrase recognition task. The evaluation includes ranking-based metrics (MRR) and cosine similarity statistics.

| Model                     | Dim  | MRR@1_para_to_orig | MRR@3_para_to_orig | MRR@5_para_to_orig | Recall@5_orig_to_para | Mean Cosine | Median Cosine | Std Dev Cosine | Min Cosine |
|----------------------------|------|-------------------|-------------------|-------------------|----------------------|-------------|---------------|----------------|------------|
| thenlper/gte-large         | 1024 | 0.95576           | 0.970813          | 0.972193          | 0.93804              | 0.958973    | 0.965250      | 0.026975       | 0.754064   |
| Qwen/Qwen3-Embedding-0.6B | 1024 | 0.95602           | 0.970983          | 0.972294          | 0.93806              | 0.903549    | 0.915874      | 0.059133       | 0.460187   |
| Qwen/Qwen3-0.6B            | 1024 | 0.65816           | 0.698337          | 0.705468          | 0.66728              | 0.952301    | 0.958187      | 0.031589       | 0.300971   |

---

#### Analysis

1. **Ranking metrics (MRR):**  
   - **thenlper/gte-large** and **Qwen/Qwen3-Embedding-0.6B** both excel in ranking the correct paraphrases, with MRR@1 ≈ 0.95 and MRR@5 > 0.97. This indicates that the first paraphrase retrieved is usually correct, and almost all correct paraphrases appear within the top 5.

   - **Qwen/Qwen3-0.6B** performs worse in ranking (MRR@1 ≈ 0.66, MRR@5 ≈ 0.71), meaning it often fails to rank the correct paraphrases at the very top.

2. **Cosine similarity statistics:**  
   - **thenlper/gte-large:**  
     Shows very high mean (0.959) and median (0.965) cosine similarities with low standard deviation (0.027). This indicates that embeddings for paraphrases are tightly clustered near their original sentence, leading to stable retrieval.

   - **Qwen/Qwen3-Embedding-0.6B:**  
     Also has good ranking metrics, but cosine similarity is lower (mean 0.904, median 0.916) with higher variance (0.059). This suggests embeddings are more spread out in vector space, but the ranking mechanism still manages to retrieve correct paraphrases.

   - **Qwen/Qwen3-0.6B:**  
     Has high cosine similarities (mean 0.952, median 0.958), but ranking performance is poor. This indicates that while the paraphrase embeddings are close to their originals, the model’s global embedding space may be less discriminative, causing unrelated sentences to sometimes appear close in cosine space.



---


In [19]:
print(embedding_complete_space)

                  Model_Name  emb_Mean_Cosine  emb_Median_Cosine  \
0         thenlper/gte-large         0.697441           0.695064   
1  Qwen/Qwen3-Embedding-0.6B         0.266335           0.261487   
2            Qwen/Qwen3-0.6B         0.818316           0.828979   

   emb_Std_Dev_Cosine  emb_Min_Cosine  emb_Max_Cosine  emb_Q1_Cosine  \
0            0.029403        0.557135             1.0       0.677500   
1            0.079906       -0.095807             1.0       0.211567   
2            0.067720       -0.064870             1.0       0.786128   

   emb_Q3_Cosine  
0       0.714620  
1       0.315185  
2       0.863603  


### Embedding Distribution Analysis

The following table reports descriptive statistics for cosine similarity distributions across embeddings. Unlike the previous analysis (focused on original–paraphrase pairs), here the evaluation considers the cosine similarity between each original phrase and all of its paraphrases.

| Model Name                | Mean Cosine | Median Cosine | Std Dev Cosine | Min Cosine | Max Cosine | Q1 Cosine | Q3 Cosine |
|----------------------------|-------------|---------------|----------------|------------|------------|-----------|-----------|
| thenlper/gte-large         | 0.697441    | 0.695064      | 0.029403       | 0.557135   | 1.000000   | 0.677500  | 0.714620  |
| Qwen/Qwen3-Embedding-0.6B | 0.266335    | 0.261487      | 0.079906       | -0.095807  | 1.000000   | 0.211567  | 0.315185  |
| Qwen/Qwen3-0.6B            | 0.818316    | 0.828979      | 0.067720       | -0.064870  | 1.000000   | 0.786128  | 0.863603  |

---

#### Analysis

1. **thenlper/gte-large:**  
   Cosine similarities are relatively high and tightly clustered, with a mean of 0.697 and a low standard deviation of 0.029. The interquartile range (0.678–0.715) is narrow, indicating consistent embeddings. The minimum similarity (0.557) is moderately high, showing that even the least similar paraphrase is fairly close in the embedding space.

2. **Qwen/Qwen3-Embedding-0.6B:**  
   Shows generally low cosine similarities (mean 0.266) with higher variability (std 0.080). The minimum cosine similarity is negative (-0.096). The interquartile range (0.212–0.315) is relatively wide, suggesting embeddings are more dispersed.

3. **Qwen/Qwen3-0.6B:**  
   Shows very high mean similarity (0.818) and median (0.829), meaning embeddings tend to cluster closely. The standard deviation (0.068) indicates moderate spread. Since the statistics consider all pairwise comparisons in the space, the high mean can partly reflect embeddings being generally close to many points, which may reduce discrimination and make retrieval harder for individual paraphrase matches.



---

### Conclusion

- **thenlper/gte-large:**  
  Excels in both ranking (MRR) and cosine similarity. High mean (0.959) and median (0.965) cosine values with low variance (0.027) indicate that paraphrase embeddings are tightly clustered around their originals, ensuring reliable retrieval.

- **Qwen/Qwen3-Embedding-0.6B:**  
  Performs well in ranking (MRR@1 ≈ 0.956) despite lower cosine similarities (mean 0.904, median 0.916) and higher variance (0.059). This suggests the model can still identify the correct paraphrases, but embeddings are more spread out in vector space.

- **Qwen/Qwen3-0.6B:**  
  Shows high cosine similarity (mean 0.952, median 0.958) but poor ranking performance (MRR@1 ≈ 0.658). This indicates that while paraphrases are close to their originals in embedding space, the global embedding distribution is less discriminative, causing unrelated sentences to sometimes appear close and reducing retrieval accuracy.

**Overall:**  
High cosine similarity alone does not guarantee strong retrieval performance. Effective paraphrase recognition requires both tight local clustering (high similarity with correct paraphrases) and sufficient global separation from unrelated sentences to achieve high ranking metrics.


---
